# Imports

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import pandas as pd
import os
import itertools
import joblib

from sklearn.preprocessing import MinMaxScaler
import keras
from keras.models import Model
from keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense, Dropout, Conv1D, BatchNormalization, MaxPooling1D, UpSampling1D
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from pathlib import Path
import seaborn as sns
from sklearn.metrics import auc

In [ ]:
TRAIN_FILEPATH = 'data_for_model/sound_levels_data.npy'
SILENT_EVENT_HOURS_1 = 'data_for_model/25_event_day.npy'
ONLY_INTRUSIONS = 'data_for_model/complete_intrusions.npy'
EVENT_TABLE_PATH = 'events_table.csv'
SCALERS_PATH = 'scalers'
MODELS_PATH = 'models'
THRESHOLDS_PATH = 'thresholds'

# Data Load Functions

In [ ]:
def load_normal_sound_normalise_per_loc(filepath):
    """Load and preprocess data of the normal day.

    This function loads the sound levels of the normal day, selects the location of interest,
    applies MinMax scaler to normalise  values per location (not global normalisation) to range of (0, 1).

    Returns: 
        numpy.ndarray: The normalized data with shape (1388, 59, 569).
        MinMaxScaler: The fitted MinMaxScaler instances per location.

    """
    # loading sound level data; shape (1388, 569, 59)
    data = np.load(filepath)

    num_positions = data.shape[1] 
    num_timesteps = data.shape[2] 
    num_samples = data.shape[0]  

    scalers = [MinMaxScaler() for _ in range(num_positions)]

    # Apply scaling separately for each position
    for i in range(num_positions):
        flat_values = data[:, i, :].reshape(-1, 1)
        scaled = scalers[i].fit_transform(flat_values)
        data[:, i, :] = scaled.reshape(num_samples, num_timesteps)

    X_train_normalized = data.transpose(0, 2, 1)

    # joblib.dump(scalers, 'scalers.pkl')

    return X_train_normalized

In [ ]:
def load_normalise_one_loc_split_train_test(train_path, location):

    data = np.load(train_path)

    loc = int((location - 1260)/10)
    data = data[:, loc, :]

    X_train, X_test = train_test_split(data, test_size=0.2, random_state=12)

    print(f"MIN:{np.min(X_train)}, MAX:{np.max(X_train)}")

    X_train_shape = X_train.shape
    X_test_shape = X_test.shape

    X_train_flat_values = X_train.reshape(-1, 1)
    X_test_flat_values = X_test.reshape(-1, 1)

    scaler = MinMaxScaler()
    X_train_normalised = scaler.fit_transform(X_train_flat_values)
    X_test_normalised = scaler.transform(X_test_flat_values)
    
    X_train_normalised = X_train_normalised.reshape(X_train_shape[0], X_train_shape[1], 1)
    X_test_normalised = X_test_normalised.reshape(X_test_shape[0], X_test_shape[1], 1)

    # joblib.dump(scaler, f'scalers/scaler_{location}.pkl')

    return X_train_normalised, X_test_normalised, scaler

In [ ]:
def load_normalise_one_loc(train_path, location, scaler = None):

    data = np.load(train_path)

    loc = int((location - 1260)/10)
    data = data[:, loc, :]
    shape = data.shape
    flat_values = data.reshape(-1, 1)
    print(f"MIN:{np.min(flat_values)}, MAX:{np.max(flat_values)}")

    if scaler is not None:
        X_normalised = scaler.transform(flat_values)

    else:
        scaler = MinMaxScaler()
        X_normalised = scaler.fit_transform(flat_values)
        
    X_normalised = X_normalised.reshape(shape[0], shape[1], 1)


    return X_normalised, scaler

In [ ]:
def load_test_one_loc(test_path, scaler, location, intrusions_only=False):

    loc = int((location - 1260)/10)
    data = np.load(test_path, allow_pickle=True)
    X_data = np.stack(data['data'], axis=0)

    if intrusions_only:
        X_file_refs = np.column_stack((data['row'], data['poi']))
        X_data = X_data[:, loc, :]
    else:
        X_file_refs = data['filename']
        if X_data.ndim == 4:
            X_data = X_data[:, loc, :, :]
        elif X_data.ndim == 3:
            X_data = X_data[:, loc, :]
    
    shape = X_data.shape

    flat_values = X_data.reshape(-1, 1)
    X_normalised = scaler.transform(flat_values)
    X_normalised = X_normalised.reshape(shape[0], shape[1], 1)

    return X_normalised, X_file_refs

# Building the Model

In [ ]:
def build_model(X_train_shape, hidden_units, dropout, lr, loss, activation_func):
    """Building of the model. 

    This function builds an LSTM-Autoencoder model based on the specified hyperparemeters.

    Args:
        X_train_shape (tuple): Shape of the input training data (batch_size, timesteps, features).
        hidden_units (int): Number of LSTM units in the encoder and decoder.
        dropout (float): Dropout rate to apply after LSTM layers.
        lr (float): Learning rate for the optimizer.
    loss (str): Loss function to be used for training.

    Returns:
        tensorflow.keras.Model: Compiled LSTM-Autoencoder model.
    
    """

    input_shape = (X_train_shape[1],X_train_shape[2])  # (time_steps, features)

    # Encoder
    input_seq = Input(shape=input_shape)
    encoded = LSTM(hidden_units, activation=activation_func, return_sequences=True)(input_seq) 
    encoded = LSTM(hidden_units//2, activation=activation_func, return_sequences=False)(encoded)
    encoded = Dropout(dropout)(encoded)

    # Latent space
    latent_space = Dense(hidden_units, activation='relu')(encoded)

    # Decoder
    decoded = RepeatVector(input_shape[0])(latent_space)  
    decoded = LSTM(hidden_units//2, activation=activation_func, return_sequences=True)(decoded)
    decoded = LSTM(hidden_units, activation=activation_func, return_sequences=True)(decoded)
    decoded = Dropout(dropout)(decoded)

    decoded = TimeDistributed(Dense(input_shape[1]))(decoded)

    model = Model(input_seq, decoded)

    model.compile(optimizer=Adam(learning_rate=lr), loss=loss)

    model.summary()

    return model

# Plots Functions

In [ ]:
def plot_original_reconstructed_one_loc(location, original_series, reconstructed_series, save_path=None):

    plt.figure(figsize=(12, 5))
    plt.plot(original_series, label="Original", linestyle="--", marker="o", color="royalblue")
    plt.plot(reconstructed_series, label="Reconstructed", linestyle="-", marker="s", color="darkorange")

    plt.xlabel("Time Step")
    plt.ylabel("Feature Value")
    plt.ylim((0, 1))
    plt.title(f"Original vs. Reconstructed Series (Location {location})")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()  
    else:
        plt.show()

In [ ]:
def error_distribution(flat_reconstruction_error):

    plt.figure(figsize=(8, 5))
    plt.hist(flat_reconstruction_error, bins=50, color="royalblue", edgecolor="black", alpha=0.75)
    plt.xlabel("Reconstruction Error", fontsize=12)
    plt.ylabel("Frequency", fontsize=12)
    plt.title("Distribution of Reconstruction Errors", fontsize=14)
    plt.grid(axis="y", linestyle="--", alpha=0.6)
    plt.show()

In [ ]:
def save_anomaly_plots(original, reconstructed, sample_indices, X_test_files, save_dir, title_prefix=""):

    os.makedirs(save_dir, exist_ok=True)

    for sample in sample_indices:
        event_id = X_test_files[sample, 0]
        locations = X_test_files[sample, 1].replace(" ", "")  

        safe_locations = locations.replace(",", "_")
        filename = f"{title_prefix.lower().replace(' ', '_')}_id_{event_id}_loc_{safe_locations}.png"

        plt.figure(figsize=(12, 5))
        plt.plot(original[sample, :, :], label='Original', linestyle="--", marker="o", color="royalblue")
        plt.plot(reconstructed[sample, :, :], label="Reconstructed", linestyle="-", marker="s", color="darkorange")
        plt.xlabel("Time Step")
        plt.ylabel("Feature Value")
        plt.title(f"Original vs. Reconstructed Series")
        plt.legend()
        plt.ylim((0, 1))
        plt.grid(True, linestyle="--", alpha=0.6)

        plt.savefig(os.path.join(save_dir, filename),  bbox_inches='tight')
        plt.close()

In [ ]:
def plot_confusion_matrix(tp, fp, fn, tn, title='Confusion Matrix'):
    cm = np.array([[tp, fn],
                   [fp, tn]])

    labels = ['Positive', 'Negative']

    plt.figure(figsize=(6, 5))
    ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                     xticklabels=labels, yticklabels=labels,
                     linewidths=0.5, linecolor='gray', square=True)


    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title, fontsize=14, weight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_roc_curves(*roc_results):
    plt.figure(figsize=(6, 6))
    
    for result in roc_results:
        label = f"{'Harsh' if result['harsh_testing'] else 'Soft'} Testing (AUC = {result['roc_auc']:.2f})"
        plt.plot(result['fprs'], result['tprs'], lw=2, label=label)
    
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves')
    plt.legend(loc="lower right")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()


# Anomaly Detection Functions

In [ ]:
def detect_anomalies_intrusions(absolute_errors, reconstructed, original, threshold, X_test_files, position, harsh_testing):
    per_event_meta = {}
    save_folder = "plots"

    for i, (max_error, meta_row) in enumerate(zip(absolute_errors, X_test_files)):
        event_id = meta_row[0]
        locations = [loc.strip() for loc in meta_row[1].split(',')]
        ground_truth = str(position) in locations

        # Initialize event if it doesn't exist
        if event_id not in per_event_meta:
            per_event_meta[event_id] = {
                "GT": int(ground_truth), 
                "Sample_IDs": set(),
                "Detected_Counter": 0
            }

        per_event_meta[event_id]["Sample_IDs"].add(i)

        # Detection logic
        if max_error > threshold:
            per_event_meta[event_id]["Detected_Counter"] += 1

            if ground_truth == False:
                if harsh_testing:
                    print(f"[FP] Event {event_id}")
                    filename = f"{save_folder}/false_positives/FP_{event_id}_pos_{position}_sample_{i}.png"
                    plot_original_reconstructed_one_loc(position, original[i], reconstructed[i], filename)
                else:
                    print(f"[SOFT TP] — Event {event_id}")
                    filename = f"{save_folder}/soft_true_positives/new_STP_{event_id}_pos_{position}_sample_{i}.png"
                    plot_original_reconstructed_one_loc(position, original[i], reconstructed[i], filename)

    # Find false negatives: ground truth = 1, but no detections
    for event_id, meta in per_event_meta.items():
        if (meta["GT"] == 1) and (meta["Detected_Counter"] == 0):
            print(f"[FN] Event {event_id}")
            for sid in meta["Sample_IDs"]:
                filename = f"{save_folder}/false_negatives/new_FN_{event_id}_pos_{position}_sample_{i}.png"
                plot_original_reconstructed_one_loc(position, original[sid], reconstructed[sid], filename)

    return per_event_meta



In [ ]:
def detect_anomalies_normal(location, absolute_errors, reconstructed, original, threshold, X_test_files, plot=False):

    save_folder = "plots"
    os.makedirs(save_folder, exist_ok=True)

    for i, max_error in enumerate(absolute_errors):
        if max_error > threshold:
            
            if X_test_files is not None:
                print(f"Anomaly detected at sample {i} {X_test_files[i]}")
                filename = f"{save_folder}/false_positives/new_FP_{X_test_files[i]}_pos_{location}.png"
            else: 
                print(f"Anomaly detected at sample {i}")
                filename = f"{save_folder}/false_positives/new_FP_silent_{i}_pos_{location}.png"
            if plot:
                plot_original_reconstructed_one_loc(location, original[i], reconstructed[i], filename)
    

# Statistics/Evaluation

In [ ]:
def get_performance(metadata, threshold, harsh_testing=True):
    TP, TN, FP, FN = 0, 0, 0, 0


    for _, row in metadata.iterrows():
        detected = row['maxAE'] > threshold
        true_label = row['true_label']
        includes_location = int(row['includes_location'])

        if (true_label == 1) and (includes_location == 1):
            if detected:
                TP += 1
            else:
                FN += 1
        
        elif (true_label) == 1 and (includes_location == 0):
            if detected:
                if harsh_testing:
                    FP += 1
                else:
                    TP +=1
            else:
                TN += 1
        
        elif true_label == 0:
            if detected: 
                FP +=1
            else:
                TN +=1


    return TP, FP, FN, TN


In [ ]:
def compute_roc_auc(metadata, train_errors, harsh_testing=True):
    tpr_list = []
    fpr_list = []

    thresholds = np.percentile(train_errors, np.linspace(0, 100, 1000))

    for threshold in thresholds:
        TP, FP, FN, TN = get_performance(metadata, threshold, harsh_testing=harsh_testing)
        TPR = TP / (TP + FN) if (TP + FN) != 0 else 0
        FPR = FP / (FP + TN) if (FP + TN) != 0 else 0
        tpr_list.append(TPR)
        fpr_list.append(FPR)

    roc_auc = auc(fpr_list, tpr_list)

    youden_index = np.array(tpr_list) - np.array(fpr_list)
    optimal_threshold_index = np.argmax(youden_index)
    optimal_threshold = thresholds[optimal_threshold_index]
    
    optimal_fpr = np.array(fpr_list)[optimal_threshold_index]
    optimal_tpr = np.array(tpr_list)[optimal_threshold_index]

    return {
        'roc_auc': roc_auc,
        'thresholds': thresholds,
        'fprs': fpr_list,
        'tprs': tpr_list,
        'harsh_testing': harsh_testing,
        'optimal_threshold': optimal_threshold,
        'optimal_fpr': optimal_fpr,
        'optimal_tpr': optimal_tpr,
    }

In [ ]:
def evaluation_per_label(meta_table, event_table_path, threshold):

    per_label_meta = {}
    # load the event table
    event_table = pd.read_csv(event_table_path)
    # loop trhough meta table
    for idx, row in meta_table.iterrows():

        # get the event id/ row
        id = row['event']
        # get the row, extract the label
        label = event_table.iloc[int(id)]['label_anon']
        # create a data frame per label with happened/detected
        if label not in per_label_meta:
            per_label_meta[label] = {
                "Happened": 1,     
                "Detected": 0
            }
        else: 
            per_label_meta[label]["Happened"] += 1

        # if detected, detected +1
        if row['maxAE'] > threshold:
            per_label_meta[label]["Detected"] += 1

    labels = list(per_label_meta.keys())
    detected = [per_label_meta[label]["Detected"] for label in labels]  # True Positives
    missed = [per_label_meta[label]["Happened"] - per_label_meta[label]["Detected"] for label in labels]  # False Negatives

    x = range(len(labels))

    plt.figure(figsize=(10, 5))
    plt.bar(x, detected, label="True Positives", color="royalblue")
    plt.bar(x, missed, bottom=detected, label="False Negatives", color="darkorange")
    plt.xticks(x, labels, rotation=45)
    plt.ylabel("Sample Count")
    plt.title("Detection Performance per Label")
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
        

# Data Load

In [ ]:
LOCATION = 4220
X_train, X_test, scaler = load_normalise_one_loc_split_train_test(TRAIN_FILEPATH, LOCATION)

# Model Load

In [ ]:
model = keras.models.load_model(f"{MODELS_PATH}/location_{LOCATION}.keras")
model.summary()

# Model Training

In [ ]:
hidden_units = 512
learning_rate = 0.001
dropout_rate = 0.2
loss = 'mae'
activation_function = 'tanh'

model = build_model(X_train.shape, hidden_units, dropout_rate, learning_rate, loss, activation_function)

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
model.fit(X_train, X_train, epochs=150, batch_size=32, validation_split=0.2, callbacks=[early_stopping])

# Model Save

In [ ]:
model.save(f"{MODELS_PATH}/location_{LOCATION}.keras")

# Performance Inspection 

In [ ]:
decoded = model.predict(X_train)

In [ ]:
sample_id = 5
plot_original_reconstructed_one_loc(LOCATION, X_train[sample_id], decoded[sample_id])

In [ ]:
train_error = np.abs(decoded - X_train)

In [ ]:
print(f"Mean: {np.mean(train_error)}, Min: {np.min(train_error)}, Max: {np.max(train_error)}, 99th Percentile: {np.percentile(train_error, 99)}")
error_distribution(train_error.flatten())

# Testing

## Silent Data (Normal Day)

### Baseline Approach

In [ ]:
anomalies_idx = np.where(X_test > 1)
[len(a) for a in anomalies_idx]

### Model 

In [ ]:
decoded_X_test = model.predict(X_test)

In [ ]:
sample_id = 20
plot_original_reconstructed_one_loc(LOCATION, X_test[sample_id], decoded_X_test[sample_id])

In [ ]:
X_test_error = abs(X_test - decoded_X_test)
X_test_error_max = np.max(X_test_error, axis=1).squeeze()
X_test_error_max.shape

In [ ]:
print(f"Mean: {np.mean(X_test_error)}, Max: {np.max(X_test_error)}, 99th Percentile: {np.percentile(X_test_error, 99)}")
error_distribution(X_test_error.flatten())

## Silent Data (Intrusion Day_1)

In [ ]:
test_data, X_test_files = load_test_one_loc(SILENT_EVENT_HOURS_1, scaler, LOCATION)
silent_test = test_data[:420, :, :]
silent_X_test_files = X_test_files[:420]

### Baseline Approach

In [ ]:
anomalies_idx = np.where(silent_test > 1)
[len(a) for a in anomalies_idx]

### Model 

In [ ]:
decoded_test_silent = model.predict(silent_test)

In [ ]:
sample_id = 200
plot_original_reconstructed_one_loc(LOCATION, silent_test[sample_id], decoded_test_silent[sample_id])

In [ ]:
silent_test_error = abs(silent_test - decoded_test_silent)
silent_test_error_max = np.max(silent_test_error, axis=1).squeeze()
silent_test_error_max.shape

In [ ]:
print(f"Mean: {np.mean(silent_test_error)}, Max: {np.max(silent_test_error)}, 99th Percentile: {np.percentile(silent_test_error, 99)}")
error_distribution(silent_test_error.flatten())

In [ ]:
np.percentile(np.concatenate([silent_test_error, X_test_error]), 99)

## Intrusion Data

In [ ]:
intrusion_test_data, event_row = load_test_one_loc(ONLY_INTRUSIONS, scaler, LOCATION, True)

### Baseline Approach

In [ ]:
anomalies_idx_intrusions = np.where(intrusion_test_data > 1)
[len(a) for a in anomalies_idx_intrusions]

### Model

In [ ]:
decoded_test_intrusion = model.predict(intrusion_test_data)

In [ ]:
sample_id = 400
plot_original_reconstructed_one_loc(LOCATION, intrusion_test_data[sample_id], decoded_test_intrusion[sample_id])

In [ ]:
intrusion_test_error = abs(intrusion_test_data - decoded_test_intrusion)

In [ ]:
max_intrusion_test_error = np.max(intrusion_test_error, axis=1).squeeze()
max_intrusion_test_error.shape

In [ ]:
print(f"Mean: {np.mean(intrusion_test_error)}, Max: {np.max(intrusion_test_error)}, 99th Percentile: {np.percentile(intrusion_test_error, 99)}")
error_distribution(intrusion_test_error.flatten())

# AUC ROC Method on Combined Data

## Combine Data

In [ ]:
X_test_meta_data = pd.DataFrame({'event': np.array([f"Silent_sample_{i}" for i in range(X_test_error_max.shape[0])]), 
                                 'maxAE': X_test_error_max, 
                                 'true_label': np.zeros(X_test_error_max.shape, dtype=int), 
                                 'includes_location': np.ones(X_test_error_max.shape, dtype=int)})

In [ ]:
silent_meta_data = pd.DataFrame({'event': silent_X_test_files, 
                                 'maxAE': silent_test_error_max, 
                                 'true_label': np.zeros(silent_test_error_max.shape, dtype=int),
                                 'includes_location': np.ones(silent_test_error_max.shape, dtype=int)})

In [ ]:
event_row_new = np.copy(event_row)
event_row_new[:, 1] = (np.char.find(event_row[:, 1], str(LOCATION)) != -1).astype(int)
intrusions_meta_data = pd.DataFrame({'event': event_row_new[:, 0], 
                                    'maxAE': max_intrusion_test_error, 
                                    'true_label': np.ones(max_intrusion_test_error.shape, dtype=int), 
                                    'includes_location': event_row_new[:, 1].astype(int)})
intrusions_meta_data = intrusions_meta_data.groupby(['event'], as_index=False).max()

In [ ]:
combined_data_frame = pd.concat([X_test_meta_data, silent_meta_data, intrusions_meta_data], ignore_index=True)

## Evaluation

In [ ]:
roc_result_harsh = compute_roc_auc(combined_data_frame, train_error, True)
roc_result_soft = compute_roc_auc(combined_data_frame, train_error, False)

In [ ]:
plot_roc_curves(roc_result_harsh, roc_result_soft)

In [ ]:
threshold_soft = roc_result_soft['optimal_threshold']
threshold_harsh = roc_result_harsh['optimal_threshold']
print(threshold_soft, threshold_harsh)

In [ ]:
np.savez(f"thresholds_{LOCATION}.npz", soft=threshold_soft, harsh=threshold_harsh)

In [ ]:
TP, FP, FN, TN = get_performance(combined_data_frame, threshold_soft, False)
plot_confusion_matrix(TP, FP, FN, TN)

# Testing on a Nearby Location

In [ ]:
TESTING_LOC = 4230
# load random 19/02 files
silent_day_t_loc, _ = load_normalise_one_loc(TRAIN_FILEPATH, TESTING_LOC, scaler)
_, silent_day_t_loc = train_test_split(silent_day_t_loc, test_size=0.2, random_state=12)
decoded_silent_day_t_loc = model.predict(silent_day_t_loc)
decoded_silent_day_t_loc_maxAE = np.max(abs(silent_day_t_loc - decoded_silent_day_t_loc), axis=1).squeeze()

# morning of 25/02
test_data_t_loc, _ = load_test_one_loc(SILENT_EVENT_HOURS_1, scaler, TESTING_LOC)
silent_morning_t_loc = test_data_t_loc[:420, :, :]
decoded_silent_morning_t_loc = model.predict(silent_morning_t_loc)
decoded_silent_morning_t_loc_maxAE = np.max(abs(silent_morning_t_loc - decoded_silent_morning_t_loc), axis=1).squeeze()

# Intrusions
intrusion_test_data_t_loc, event_row_t_loc = load_test_one_loc(ONLY_INTRUSIONS, scaler, TESTING_LOC, True)
decoded_intrusion_test_data_t_loc = model.predict(intrusion_test_data_t_loc)
decoded_intrusion_test_data_t_loc_maxAE = np.max(abs(intrusion_test_data_t_loc - decoded_intrusion_test_data_t_loc), axis=1).squeeze()

In [ ]:
error_distribution(np.concatenate([abs(silent_day_t_loc - decoded_silent_day_t_loc), abs(silent_morning_t_loc - decoded_silent_morning_t_loc)]).flatten())

In [ ]:
silent_day_t_loc_meta_data = pd.DataFrame({'event': np.array([f"Silent_d_{i}" for i in range(decoded_silent_day_t_loc_maxAE.shape[0])]), 
                                 'maxAE': decoded_silent_day_t_loc_maxAE, 
                                 'true_label': np.zeros(decoded_silent_day_t_loc_maxAE.shape[0], dtype=int), 
                                 'includes_location': np.ones(decoded_silent_day_t_loc_maxAE.shape[0], dtype=int)})

silent_morning_t_loc_meta_data = pd.DataFrame({'event': np.array([f"Silent_m_{i}" for i in range(decoded_silent_morning_t_loc_maxAE.shape[0])]), 
                                 'maxAE': decoded_silent_morning_t_loc_maxAE, 
                                 'true_label': np.zeros(decoded_silent_morning_t_loc_maxAE.shape[0], dtype=int),
                                 'includes_location': np.ones(decoded_silent_morning_t_loc_maxAE.shape[0], dtype=int)})

event_row_new = np.copy(event_row_t_loc)
event_row_new[:, 1] = (np.char.find(event_row_t_loc[:, 1], str(TESTING_LOC)) != -1).astype(int)
intrusion_t_loc_meta_data = pd.DataFrame({'event': event_row_new[:, 0], 
                                    'maxAE': decoded_intrusion_test_data_t_loc_maxAE, 
                                    'true_label': np.ones(decoded_intrusion_test_data_t_loc_maxAE.shape[0], dtype=int), 
                                    'includes_location': event_row_new[:, 1].astype(int)})
intrusion_t_loc_meta_data = intrusion_t_loc_meta_data.groupby(['event'], as_index=False).max()

In [ ]:
combined_t_loc_meta_data = pd.concat([silent_day_t_loc_meta_data, silent_morning_t_loc_meta_data, intrusion_t_loc_meta_data], ignore_index=True)

In [ ]:
thresholds = np.load(f'{THRESHOLDS_PATH}/thresholds_{LOCATION}.npz')

In [ ]:
TP, FP, FN, TN = get_performance(combined_t_loc_meta_data, thresholds['soft'], False)
plot_confusion_matrix(TP, FP, FN, TN)

# Anomaly Detection

In [ ]:
meta_peformance = detect_anomalies_intrusions(max_intrusion_test_error, decoded_test_intrusion, intrusion_test_data, threshold_soft, event_row, LOCATION, False)

In [ ]:
detect_anomalies_normal(LOCATION, silent_test_error_max, decoded_test_silent, silent_test, threshold_soft, silent_X_test_files, plot=True)

In [ ]:
evaluation_per_label(intrusions_meta_data, EVENT_TABLE_PATH, threshold_soft)

# The Best Model Find

In [ ]:
# for model in model_directory
for model_name in os.listdir(MODELS_PATH):
    # if the name does not include LOCATION, continue
    if str(LOCATION) not in model_name:
        continue

    # else: load the model, print the name
    model = keras.models.load_model(os.path.join(MODELS_PATH, model_name))
    print(f"Working with the model: {model_name}")
    decoded = model.predict(X_train)
    train_error = np.abs(decoded - X_train)
    
    # predict on X_test, predict on Normal hours (25), predict on Intrusions
    X_test_decoded = model.predict(X_test)
    X_test_error = abs(X_test - X_test_decoded)
    X_test_error_max = np.max(X_test_error, axis=1).squeeze()

    decoded_test_silent = model.predict(silent_test)
    silent_test_error = abs(silent_test - decoded_test_silent)
    silent_test_error_max = np.max(silent_test_error, axis=1).squeeze()

    decoded_test_intrusion = model.predict(intrusion_test_data)
    intrusion_test_error = abs(intrusion_test_data - decoded_test_intrusion)
    max_intrusion_test_error = np.max(intrusion_test_error, axis=1).squeeze()
    

    # make data_frame
    X_test_meta_data = pd.DataFrame({'event': np.array([f"Silent_sample_{i}" for i in range(X_test_error_max.shape[0])]), 
                                 'maxAE': X_test_error_max, 
                                 'true_label': np.zeros(X_test_error_max.shape, dtype=int), 
                                 'includes_location': np.ones(X_test_error_max.shape, dtype=int)})
                                 
    silent_meta_data = pd.DataFrame({'event': silent_X_test_files, 
                                 'maxAE': silent_test_error_max, 
                                 'true_label': np.zeros(silent_test_error_max.shape, dtype=int),
                                 'includes_location': np.ones(silent_test_error_max.shape, dtype=int)})
                                 
    event_row_new = np.copy(event_row)
    event_row_new[:, 1] = (np.char.find(event_row[:, 1], str(LOCATION)) != -1).astype(int)
    intrusions_meta_data = pd.DataFrame({'event': event_row_new[:, 0], 
                                    'maxAE': max_intrusion_test_error, 
                                    'true_label': np.ones(max_intrusion_test_error.shape, dtype=int), 
                                    'includes_location': event_row_new[:, 1].astype(int)})
    intrusions_meta_data = intrusions_meta_data.groupby(['event'], as_index=False).max()
    combined_data_frame = pd.concat([X_test_meta_data, silent_meta_data, intrusions_meta_data], ignore_index=True)
    
    # auc roc method
    roc_result_harsh = compute_roc_auc(combined_data_frame, train_error, True)
    roc_result_soft = compute_roc_auc(combined_data_frame, train_error, False)
    plot_roc_curves(roc_result_harsh, roc_result_soft)

    threshold_soft = roc_result_soft['optimal_threshold']
    threshold_harsh = roc_result_harsh['optimal_threshold']
    # confusion matrices

    TP_h, FP_h, FN_h, TN_h = get_performance(combined_data_frame, threshold_harsh, True)
    TP_s, FP_s, FN_s, TN_s = get_performance(combined_data_frame, threshold_soft, False)
    plot_confusion_matrix(TP_h, FP_h, FN_h, TN_h)
    plot_confusion_matrix(TP_s, FP_s, FN_s, TN_s)

